# c14.co.il — aio-login namespace anonymous-access check

Follow-up on the custom `aio-login` REST namespace discovered via
`GET /wp-json/` (likely a login-hardening plugin: URL renaming,
reCAPTCHA, login-attempt limiting, login-page branding). GET/OPTIONS
only — no POST/PUT/DELETE is ever sent, even to routes that advertise
those methods via OPTIONS. If a route both returns 200 on GET *and*
advertises POST/PUT, that's noted for a separately-authorized
follow-up, not acted on here.

Routes whose names imply admin/sensitive functionality
(`change-wp-admin-login`, `dashboard/update`, `dashboard/logs`,
`grecaptcha`, `logs`, `limit-login-attempts`) are flagged as HIGH
CONCERN if reachable without authentication.

Uses the same owner-configured WAF-bypass header as
`colab_recon.ipynb` (Colab Secret `PENTEST_BYPASS_TOKEN_C14`) since
this target sits behind a WAF.

**Only run this against a target you are authorized to test.**


In [ ]:
TARGET = "https://c14.co.il"  #@param {type:"string"}
DELAY = 1.5  #@param {type:"number"}
TIMEOUT = 10.0  #@param {type:"number"}

# Optional WAF-bypass header, for a Cloudflare custom rule YOU configure
# yourself (e.g. "skip security if header X-Pentest-Token equals
# <secret>"). Do NOT put the secret value directly in the field below if
# you ever save/commit this notebook -- use Colab's built-in Secrets
# instead (key icon in the left sidebar: add a secret named
# PENTEST_BYPASS_TOKEN_C14, grant this notebook access). The manual field is
# only a fallback for quick one-off runs you don't save afterward.
BYPASS_HEADER_NAME = "X-Pentest-Token"  #@param {type:"string"}
BYPASS_HEADER_VALUE_MANUAL = ""  #@param {type:"string"}

import json
import re
import time
import urllib.error
import urllib.request

USER_AGENT = "pentest-c14-colab-recon/1.0 (+authorized-assessment)"

BYPASS_HEADER_VALUE = BYPASS_HEADER_VALUE_MANUAL
try:
    from google.colab import userdata
    _secret = userdata.get("PENTEST_BYPASS_TOKEN_C14")
    if _secret:
        BYPASS_HEADER_VALUE = _secret
except Exception:
    pass  # not in Colab, or no secret configured -- fall back to manual field above

if BYPASS_HEADER_NAME and BYPASS_HEADER_VALUE:
    print(f"WAF-bypass header configured: {BYPASS_HEADER_NAME} (value hidden)")

def fetch(url, timeout=TIMEOUT, method="GET", follow_redirects=True):
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **kw):
            return None
    opener = (urllib.request.build_opener(NoRedirect)
              if not follow_redirects else urllib.request.build_opener())
    req_headers = {"User-Agent": USER_AGENT}
    if BYPASS_HEADER_NAME and BYPASS_HEADER_VALUE:
        req_headers[BYPASS_HEADER_NAME] = BYPASS_HEADER_VALUE
    req = urllib.request.Request(url, headers=req_headers, method=method)
    try:
        with opener.open(req, timeout=timeout) as resp:
            return resp.status, dict(resp.headers), resp.read()
    except urllib.error.HTTPError as e:
        return e.code, dict(e.headers or {}), e.read()
    except urllib.error.URLError as e:
        return None, {}, str(e).encode()

base = TARGET.rstrip("/")
print(f"Target: {base}")

# A blanket 401/403/429 across nearly every path (including harmless
# ones) usually means a WAF/bot-protection layer is blocking on
# User-Agent or source IP/ASN (e.g. Colab's Google Cloud IP range),
# not real per-path access control.
WAF_SIGNATURES = {
    "cloudflare": ["cloudflare", "attention required", "cf-ray", "cf-mitigated"],
    "sucuri": ["sucuri website firewall", "sucuri/cloudproxy"],
    "imperva/incapsula": ["incapsula", "imperva", "_incap_ses"],
    "akamai": ["akamai", "ak_bmsc"],
    "aws-waf": ["aws waf", "x-amzn-requestid", "awselb"],
    "generic-block-page": ["access denied", "request blocked", "bot detected"],
}

def waf_hint(status, headers, body):
    if status not in (401, 403, 429):
        return None
    blob = (body[:2000].decode("utf-8", errors="replace") + " "
            + " ".join(f"{k}:{v}" for k, v in headers.items())).lower()
    for vendor, needles in WAF_SIGNATURES.items():
        if any(n in blob for n in needles):
            return vendor
    return "unknown-vendor"

waf_request_count = 0
waf_blocked_count = 0
waf_vendors_seen = set()

def track_waf(status, headers, body):
    global waf_request_count, waf_blocked_count
    waf_request_count += 1
    hint = waf_hint(status, headers, body)
    if hint:
        waf_blocked_count += 1
        waf_vendors_seen.add(hint)
    return hint


In [ ]:
ROUTES = {
    "dashboard": "/wp-json/aio-login/dashboard",
    "dashboard/update": "/wp-json/aio-login/dashboard/update",
    "dashboard/logs": "/wp-json/aio-login/dashboard/logs",
    "change-wp-admin-login": "/wp-json/aio-login/change-wp-admin-login",
    "grecaptcha": "/wp-json/aio-login/grecaptcha",
    "limit-login-attempts": "/wp-json/aio-login/limit-login-attempts",
    "logs": "/wp-json/aio-login/logs",
    "custom-css": "/wp-json/aio-login/custom-css",
    "background": "/wp-json/aio-login/background",
    "logo": "/wp-json/aio-login/logo",
}

# Routes whose names imply sensitive data or an admin action -- flag
# loudly if reachable without auth.
HIGH_CONCERN = {
    "dashboard/update", "dashboard/logs", "change-wp-admin-login",
    "grecaptcha", "logs", "limit-login-attempts",
}

# Precise WAF-signature check (unlike the shared track_waf() from the
# setup cell, this does NOT treat every plain 401/403 as a WAF hit --
# a 401 here is the *expected good* outcome for a protected route, not
# a block. Only flag when an actual vendor signature is present.
def precise_waf_hint(status, headers, body):
    if status not in (401, 403, 429):
        return None
    blob = (body[:2000].decode("utf-8", errors="replace") + " "
            + " ".join(f"{k}:{v}" for k, v in headers.items())).lower()
    for vendor, needles in WAF_SIGNATURES.items():
        if any(n in blob for n in needles):
            return vendor
    return None

exposed = []
for name, path in ROUTES.items():
    url = base + path

    status, headers, resp_body = fetch(url, method="GET")
    hint = precise_waf_hint(status, headers, resp_body)
    flag = ""
    if status == 200:
        flag = " <-- EXPOSED without auth" + (", HIGH CONCERN" if name in HIGH_CONCERN else "")
        exposed.append(name)
    elif hint:
        flag = f" [WAF/bot-protection block? {hint}]"
    print(f"[{status}] GET  {url}{flag}")
    if status == 200:
        snippet = resp_body[:200].decode("utf-8", errors="replace").replace("\n", " ")
        print(f"    body preview: {snippet!r}")
    time.sleep(DELAY)

    opt_status, opt_headers, _ = fetch(url, method="OPTIONS")
    allow = opt_headers.get("Allow", "n/a")
    print(f"[{opt_status}] OPTIONS {url} -> Allow: {allow}")
    if any(m in allow for m in ("POST", "PUT", "DELETE")):
        print("    ** write method(s) advertised -- NOT invoked here, note for a "
              "separately-authorized follow-up if GET above was also exposed **")
    time.sleep(DELAY)

print("\n== Summary ==")
if exposed:
    print(f"{len(exposed)} route(s) returned 200 without authentication: {', '.join(exposed)}")
    high = [n for n in exposed if n in HIGH_CONCERN]
    if high:
        print(f"HIGH CONCERN (name implies admin/sensitive data): {', '.join(high)} -- "
              f"investigate manually, do not assume severity from route name alone.")
else:
    print("No aio-login routes were reachable without authentication.")
